In [1]:
import pandas as pd
import pandas

In [1]:

# df['num_redundant_layers']

In [2]:
%matplotlib inline
from numpy import array, linspace ; from numpy.random import randint
from matplotlib.pyplot import hist, xticks, show
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
# synthesize some data 
font = {
    'family' : 'serif',
    'weight':'normal',
        'size'   : 12}

matplotlib.rc('font', **font)
import seaborn as sns
pallete = sns.color_palette("Set1", 4)
dgl_color = pallete[0]
quiver_color = pallete[1]
split_color = pallete[2]
p3_color = pallete[3]
hatches = list('-*/o')

pallete = sns.color_palette("Set1", 5)
pallete

[(0.8941176470588236, 0.10196078431372549, 0.10980392156862745),
 (0.21568627450980393, 0.49411764705882355, 0.7215686274509804),
 (0.30196078431372547, 0.6862745098039216, 0.2901960784313726),
 (0.596078431372549, 0.3058823529411765, 0.6392156862745098),
 (1.0, 0.4980392156862745, 0.0)]

In [ ]:
import pandas as pd
machine_name = 'nvlink'
# machine_name = 'g4dn.12xlarge/pcie'
df = pd.read_csv(f'../experiment/logs/main.csv')
# print(df)
d = {'system': 1, 'b': 2, 'c': 3}
dfs = []
idx = ['model','system']
dgl_cache_pers = [1, 0, 0, 0,]
graphs = ['ogbn-papers100M']
graphs = ['ogbn-papers100M','com-orkut','com-friendster']
graphs = ['papers100M','orkut','friendster']
graphs = ['orkut', 'papers100M', 'friendster']
# graphs = ['orkut']
# graphs = []
# , 'com-orkut', 'com-friendster']
for graph_name in graphs:
    idx.append(f'sampling_{graph_name}')
    idx.append(f'loading_{graph_name}')
    idx.append(f'training_{graph_name}')
    idx.append(f'total_{graph_name}')
batch_size = 256
system_label = {}
system_label = {'dgl':'DGL','p3':'Push-Pull', 'quiver':'Quiver',\
                    'dist_cache':'dist_cache','split-uva':'spa-uva','split':'spa'}
graph_label = {'ogbn-products':"Products", "ogbn-papers100M":"Papers100M", \
                   "com-orkut":"Orkut", "com-friendster":"Friendster"}
graph_label = {'ogbn-products':"Products", "papers100M":"Papers100M", \
                   "orkut":"Orkut", "friendster":"Friendster"}
dgl_time = {}
for dataset_id,graph_name in enumerate(graphs):
    split_time = {} 
    for i, system in enumerate(['split','dgl','p3','quiver','dist_cache','split']):
        out = {}
        out['graph_name'] = graph_label[graph_name]
        out['system'] = system_label[system]
        for model in ["sage", "gat"]:
            dgl_cache_per = dgl_cache_pers[dataset_id]
            dataset_df = df[(df['graph_name'] == graph_name) & (df['model'] == model)\
                        ]
            
            if system == "dgl":
                row = dataset_df[dataset_df['system'].str.contains('dgl')]
#                 print(row)
            if system == "dist_cache":
                row = dataset_df[dataset_df['system'].str.contains('dist_cache')]
            if system == "quiver":
            
                row = dataset_df[dataset_df['system'].str.contains('quiver')]
            if system == "split-uva":
                row = dataset_df[dataset_df['system'].str.contains('split') \
                                       & (dataset_df['partition_type'] == 'ndst_efreq_xbal') \
                        & (dataset_df['sample_mode'] == 'uva')]
                
            if system == "split":
                groot_row = dataset_df[dataset_df['system'].str.contains('split') \
                                       & (dataset_df['partition_type'] == 'ndst_efreq_xbal') & \
                                       (dataset_df['sample_mode'] == 'gpu')]
                if len(groot_row) != 1:
                    row = groot_row[~((groot_row['cache_size'] == '0') | (groot_row['cache_size'] == '1'))]
                else:
                    row = groot_row
                if(len(row) != 1):
                    row = row[row['num_redundant_layers'] == 0]
            if system == "p3":
                row = dataset_df[dataset_df['system'].str.contains('p3')]
            if len(row) != 1:
                print(row, graph_name, system)
            sampling_time = row['sampling (s)'].item()
            if sampling_time != "oom":
                n_epochs = float(row['num_epoch'].item())
                sampling_time = float(sampling_time)
                data_loading = float(row['feature (s)'].item())
                training_time = (float(row['forward (s)'].item()) + float(row['backward (s)'].item()))
                out[f'sampling_{model}'] = "{:.2f}".format(sampling_time)
                out[f'loading_{model}'] = "{:.2f}".format(data_loading)
                out[f'data_{model}'] = "{:.2f}".format(training_time)
                training_time = sampling_time + data_loading + training_time
                out[f'total_{model}'] = "{:.2f}".format(training_time)
                if system == "split":
                    split_time[model] = training_time
                    out[f'speed_up_{model}']  = ''
                else:
                    out[f'speed_up_{model}'] = '{:.1f}x'.format(training_time/split_time[model])
                    
            else:
                out[f'sampling_{model}'] = "OOM"
                out[f'loading_{model}'] = "OOM"
                out[f'data_{model}'] = "OOM"
                out[f'total_{model}'] = "OOM"
                  
        if i != 0:
            dfs.append(pd.Series(out).to_frame().T)
final = (pd.concat(dfs, ignore_index = True))
print(final.to_csv(index = False))
# print(final)
print(machine_name)

graph_name,system,sampling_sage,loading_sage,data_sage,total_sage,speed_up_sage,sampling_gat,loading_gat,data_gat,total_gat,speed_up_gat
Orkut,DGL,1.49,62.67,9.20,73.36,4.4x,1.51,62.75,17.12,81.38,3.6x
Orkut,Push-Pull,4.04,1.55,8.53,14.12,0.8x,4.04,1.65,37.62,43.31,1.9x
Orkut,Quiver,4.87,4.27,8.65,17.79,1.1x,4.71,4.20,16.59,25.50,1.1x
Orkut,dist_cache,4.59,4.33,8.66,17.58,1.1x,4.47,4.32,16.37,25.16,1.1x
Orkut,\name,1.90,0.09,14.75,16.74,,1.89,0.09,20.53,22.51,
Papers100M,DGL,4.64,9.48,11.28,25.40,1.4x,4.70,9.00,31.65,45.35,1.2x
Papers100M,Push-Pull,3.28,11.51,25.84,40.63,2.2x,3.33,11.30,65.72,80.35,2.2x
Papers100M,Quiver,11.84,10.40,11.70,33.94,1.9x,11.04,11.03,31.38,53.45,1.4x
Papers100M,dist_cache,11.12,5.96,10.93,28.01,1.5x,10.78,6.39,30.53,47.70,1.3x
Papers100M,\name,3.92,2.60,11.79,18.31,,3.79,2.32,31.09,37.20,
Friendster,DGL,62.71,283.40,61.11,407.22,2.9x,62.56,284.76,245.94,593.26,1.7x
Friendster,Push-Pull,85.90,350.78,151.47,588.15,4.1x,76.48,351.35,613.78,1041.61,3.0x
Friendst